# Errors, Exceptions, Warnings and Logging in Python

**Goal of this lesson:** understand what happens when things go wrong in a program, and how to deal with it.

By the end you will know how to:
1. Read an error message (a *traceback*) without panicking
2. Recognise the most common Python errors
3. Catch errors with `try` / `except` / `else` / `finally`
4. Create your own errors with `raise`
5. Use **warnings** for "this is suspicious, but I can continue"

---
## 1. Two kinds of errors

### 1.1 Syntax errors: "I can't even read this"
A **syntax error** happens when the code breaks the grammar rules of Python, like writing a sentence with no verb. Python refuses to run *anything* in the cell.

Run the cell below. Notice that the first `print` is never executed, because Python reads the whole cell before running it.

In [ ]:
print("This line will NOT run")   # code is fine here, but there's a syntax error in the cell
print("Hello"     # missing closing parenthesis

### 1.2 Exceptions: "I understood you, but I can't do it"
An **exception** happens while the program is running. The syntax is fine, but the action is impossible.

This time the first line **does** run, and the program stops at the line that fails.

In [ ]:
print("This line runs fine")
result = 10 / 0
print("This line is never reached")

---
## 2. How to read a traceback

When an exception happens, Python prints a **traceback**. It looks scary, but it follows a fixed structure.

**Read it from the BOTTOM up:**
1. **Last line** → the *type* of error and a message (e.g. `ZeroDivisionError: division by zero`)
2. **Lines above** → *where* it happened (file, line number, the code)
3. **Further up** → the chain of function calls that led there

Let's see a traceback that goes through several functions:

In [ ]:
def compute_average(numbers):
    return sum(numbers) / len(numbers)

def report(numbers):
    avg = compute_average(numbers)
    print("The average is", avg)

number = []
report(number)   # an empty list!

**Reading it:**
- Bottom line: `ZeroDivisionError: division by zero` → *what* happened
- Just above: the line `return sum(numbers) / len(numbers)` → *where* it happened
- Above that: `report([])` called `compute_average` → *how we got there*

The real cause is that we passed an empty list, so `len(numbers)` is `0`.

---
## 3. The most common exceptions

You will meet these over and over. Run each cell, read the last line of the traceback, and see how the name of the error already describes the problem.

| Exception | Meaning in plain words |
|---|---|
| `ZeroDivisionError` | You divided by zero |
| `ValueError` | Right type, but the value makes no sense (`int("hello")`) |
| `TypeError` | Wrong type for the operation (`"5" + 3`) |
| `NameError` | You used a variable that doesn't exist (often a typo) |
| `IndexError` | List position doesn't exist |
| `KeyError` | Dictionary key doesn't exist |
| `AttributeError` | The object doesn't have that method or attribute |
| `FileNotFoundError` | The file you tried to open isn't there |

In [ ]:
# ValueError: the type is right (a string), but its content can't become a number
int("hello")

In [ ]:
# TypeError: you can't add text and a number
"Age: " + 30

In [ ]:
# NameError: typo in the variable name
student_name = "Anna"
print(studnet_name)

In [ ]:
# IndexError: the list has 3 elements, at positions 0, 1, 2
grades = [28, 30, 25]
grades[3]

In [ ]:
# KeyError: the key does not exist in the dictionary
ages = {"Anna": 21, "Marco": 23}
ages["Luca"]

In [ ]:
# AttributeError: lists don't have a method called 'push' (it's 'append')
grades = [28, 30]
grades.push(25)

In [ ]:
# FileNotFoundError: the file doesn't exist
open("file_that_does_not_exist.txt")

💡 **All these errors are related.** They are all "children" of a general class called `Exception`. This matters later: catching `Exception` catches almost everything, which is usually *too much*.

---
## 4. Catching exceptions: `try` / `except`

So far every error **stopped the program**. Often we want to *handle* the problem instead, for example by asking the user again or using a default value.

```python
try:
    # code that might fail
except SomeError:
    # what to do if it fails
```


In [ ]:
user_input = "abc"   # imagine the user typed this

try:
    age = int(user_input)
    print("You are", age, "years old")
except ValueError:                                # if try fails, except will be called, otherwise it will be skipped
    print("That's not a valid number!")

print("The program continues normally.")

### 4.1 Getting the error message with `as`
`except ValueError as e` stores the error in the variable `e`, so you can show or record the message.

In [ ]:
try:
    int("abc")
except ValueError as e:
    print("Something went wrong:", e)
    print("Error type:", type(e).__name__)     # check the type of the exception

### 4.2 Handling different errors differently
You can have several `except` blocks. Python checks them **in order** and runs the first one that matches.

In [ ]:
def safe_divide(a, b):
    try:
        result = a / b
        print(f"{a} / {b} = {result}")
    except ZeroDivisionError:
        print("ZeroDivisionError: You can't divide by zero.")
    except TypeError:
        print("TypeError: Both values must be numbers.")

safe_divide(10, 2)   # fine
safe_divide(10, 0)   # division by zero
safe_divide(10, "two")  # type error

### 4.3 ⚠️ Don't catch everything blindly
This looks convenient but is a **bad habit**:
```python
try:
    ...
except:          # catches EVERYTHING, even typos!
    pass         # and silently ignores them
```
Look what happens when we make a typo inside the `try`:

In [ ]:
numbers = [1, 2, 3]

try:
    total = sum(nubmers)        # typo! NameError
except:
    print("The list is empty!")  # wrong explanation, the real bug is hidden

**Rule of thumb:** catch only the errors you *expect* and *know how to handle*. Let everything else crash loudly so you can find and fix it.

---
## 5. `else` and `finally`

The complete structure is:
```python
try:
    # code that might fail
except SomeError:
    # runs ONLY if that error happened
else:
    # runs ONLY if NO error happened
finally:
    # runs ALWAYS, error or not
```

`finally` is the place for *cleanup*: closing files, closing connections, releasing resources.

In [ ]:
def convert(text):
    print(f"--- converting {text!r} ---")
    try:
        number = int(text)
    except ValueError:
        print("except:  conversion failed")
    else:
        print("else:    conversion worked, number =", number)
    finally:
        print("finally: this always runs")

convert("42")
convert("forty-two")

### 5.1 `finally` runs even if the error is NOT caught
Here the error is a `ZeroDivisionError`, but we only catch `ValueError`. The program crashes, but `finally` still runs first.

In [ ]:
try:
    print("Opening connection...")
    x = 1 / 0
except ValueError:
    print("This won't run")
    print("Neither will this")
finally:
    print("Closing connection (always!)")

### 5.2 A real-world case: files
Opening a file that may not exist is a classic case for `try`/`except`.

(In everyday code you'll use `with open(...)`, which closes the file automatically, effectively a built-in `finally`.)

In [ ]:
# First, let's create a small file to work with
with open("grades.txt", "w") as f:
    f.write("28\n30\nabsent\n25\n")

def read_grades(filename):
    try:
        with open(filename) as f:
            lines = f.read().splitlines()
    except FileNotFoundError:
        print(f"File {filename!r} not found.")
        return []

    grades = []
    for line in lines:
        try:
            grades.append(int(line))
        except ValueError:
            print(f"Skipping invalid line: {line!r}")
    return grades

print(read_grades("grades.txt"))
print(read_grades("missing.txt"))

---
## 6. Raising your own errors with `raise`

So far Python raised the errors. You can raise them **yourself** when your code receives something that makes no sense.

In [ ]:
def set_age(age):
    if not isinstance(age, int):
        raise TypeError(f"Age must be an integer, got {type(age).__name__}")
    if age < 0:
        raise ValueError(f"Age cannot be negative, got {age}")
    print("Age set to", age)

set_age(25)

In [ ]:
set_age(-5)

And because `set_age` raises standard exceptions, whoever *uses* the function can catch them with `try`/`except`:

In [ ]:
for value in [30, -2, "twenty"]:
    try:
        set_age(value)
    except (ValueError, TypeError) as e:   # catch two types at once
        print("Invalid input:", e)

### 6.1 Custom exceptions
For your own projects you can create **your own exception types**. It's as simple as creating a class that inherits from `Exception`. This makes errors more meaningful and easier to catch specifically.

In [ ]:
class InsufficientFundsError(Exception):
    """Raised when a withdrawal is larger than the balance."""
    pass

def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFundsError(
            f"Cannot withdraw {amount}€, balance is only {balance}€"
        )
    return balance - amount

try:
    new_balance = withdraw(100, 250)
except InsufficientFundsError as e:
    print("Transaction refused:", e)

---
## 7. Warnings: "Careful, but I'll continue"

Not every problem should stop the program. A **warning** says: *"this looks suspicious, you should check, but I can go on."*

Common uses:
- A function is **deprecated** (will be removed in a future version)
- Data looks strange but is still usable
- A result might be imprecise

In [ ]:
import warnings

def average(numbers):
    if len(numbers) < 3:
        warnings.warn("Very few values: the average may not be meaningful")
    return sum(numbers) / len(numbers)

result = average([10, 20])
print("Result:", result)    # the program continues!

### 7.1 Warning categories
Like exceptions, warnings have types. The most common:

| Category | When to use it |
|---|---|
| `UserWarning` | Default, generic warning |
| `DeprecationWarning` | "This will be removed in the future, use something else" |
| `RuntimeWarning` | Something dubious happened while running |
| `FutureWarning` | "This behaviour will change in a future version" |

You'll see `FutureWarning` and `DeprecationWarning` often from libraries like **pandas** and **numpy**.

In [ ]:
def old_function():                           # The function does nothing and return None. Just to show how deprecation warnings are used.
    warnings.warn(
        "old_function() is deprecated, use new_function() instead",
        DeprecationWarning
    )
    return None

old_function()

### 7.2 Controlling warnings
You can **hide** warnings you already know about, or turn them **into errors** (useful while testing, to make sure nothing suspicious slips through).

The `with warnings.catch_warnings():` block keeps these changes temporary.

In [ ]:
# Hide warnings (only inside this block)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")      # ignore let us hide the warning message
    print("Silent:", average([1, 2]))
    print("I can still print this line, even if the average() function issues a warning.")

# Turn warnings into errors (only inside this block)
with warnings.catch_warnings():
    warnings.simplefilter("error")       # error converts warnings into exceptions
    average([1, 2])
    print("This line will not be printed!")

In [ ]:
# CORRECT WAY OF HANDLING ERRROS: Turn warnings into errors (only inside this block)
with warnings.catch_warnings():
    warnings.simplefilter("error")       # error converts warnings into exceptions
    try:
        average([1, 2])
    except UserWarning as w:
        print("The warning became an error:", w)
        print("This line will still be printed")

⚠️ Ignoring warnings is fine when you **understand** them. Ignoring them just because they are annoying is like covering the "low fuel" light with tape.

---
## 9. Putting it all together

A small program that reads student grades. It uses **everything** we've seen:
- `raise` to validate data
- a **custom exception**
- `try` / `except` / `else` / `finally`
- a **warning** for suspicious data

In [ ]:
import sys, warnings


class InvalidGradeError(Exception):
    """Raised when a grade is outside the allowed range."""


def parse_grade(text):
    grade = int(text)                 # may raise ValueError
    if not 18 <= grade <= 30:
        raise InvalidGradeError(f"grade {grade} is not between 18 and 30")
    return grade


def process(raw_grades):
    print(f"Processing {len(raw_grades)} entries")
    valid = []
    try:
        for raw in raw_grades:
            try:
                g = parse_grade(raw)
            except ValueError:
                print(f"Not a number, skipped: {raw!r}")
            except InvalidGradeError as e:
                print(f"Invalid grade, skipped: {e}")
            else:
                valid.append(g)

        if len(valid) < 3:
            warnings.warn("Fewer than 3 valid grades: average is unreliable")

        avg = sum(valid) / len(valid)  # may raise ZeroDivisionError
    except ZeroDivisionError:
        print("No valid grades at all, cannot compute an average")
        return None
    finally:
        print("Processing finished")

    print(f"Average: {avg:.2f}")
    return avg


process(["28", "30", "abc", "45", "25", "27"])

In [ ]:
process(["hello", "99"])

---
## 10. Summary

| Tool | What it does | When to use it |
|---|---|---|
| **Traceback** | Shows what went wrong and where | Read it bottom-up, every time |
| `try` / `except` | Catches an error and handles it | When you *expect* an error and know what to do |
| `else` | Runs if no error occurred | Code that should run only on success |
| `finally` | Always runs | Cleanup (close files, connections) |
| `raise` | Creates an error | Validate inputs, fail early |
| `warnings.warn` | Signals a problem without stopping | Suspicious data, deprecated features |

### Golden rules
1. ✅ Catch **specific** exceptions (`except ValueError`), not everything
2. ❌ Never use a bare `except:` followed by `pass`
3. ✅ Fail early with a **clear message** (`raise ValueError("...")`)
4. ✅ Use `finally` (or `with`) for cleanup
5. ✅ Don't ignore warnings you don't understand

---
## 11. Exercises

**Exercise 1.** Write a function `ask_number(text)` that converts `text` to an `int`. If the conversion fails, print a friendly message and return `0`.

**Exercise 2.** Write a function `get_item(lst, index)` that returns `lst[index]`, but returns `None` and prints a warning if the index doesn't exist.

**Exercise 3.** Write a function `set_temperature(celsius)` that raises a `ValueError` if the temperature is below -273.15 (absolute zero), and issues a **warning** if it's above 100.

**Exercise 4.** Find the bug: why does this code say "File not found" even though the problem is something else?
```python
try:
    with open("grades.txt") as f:
        data = f.read()
    total = sum(dat)
except:
    print("File not found")
```